# 💬 Чат с нашей нейросетью

🎭 **Ролевая игра:** в чате открой «Персонаж и настройки» (под окном чата) и опиши персонажа.

**Нажми ▶ у ячейки ниже** — через ~1 минуту появится ссылка `https://….gradio.live`. Открой её — это чат во весь экран.

⚠️ Бесплатный Colab **выключается, если долго ничего не делать** — тогда ссылка перестаёт работать. Просто вернись сюда, нажми ▶ ещё раз и открой **новую** ссылку (она каждый раз другая).

🧠 **Модель намного умнее (Qwen 1,5 млрд, ролевая):** [открыть colab_qwen](https://colab.research.google.com/github/r0561952-beep/ai/blob/ccr-d2dd0252-6ksj1l/colab_qwen.ipynb) — нужна видеокарта T4.

In [ ]:
#@title ▶ Запустить чат
модель = "ролевая (из GitHub)" #@param ["ролевая (из GitHub)", "обычная (из GitHub)", "большая (с Google Диска)"]
import os, sys
REPO = '/content/ai'
BRANCH = 'ccr-d2dd0252-6ksj1l'
# 1) скачиваем самую свежую версию проекта
if os.path.exists(REPO):
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
else:
    !git clone -q -b {BRANCH} https://github.com/r0561952-beep/ai.git {REPO}
# 2) ставим библиотеки
!pip install -q tokenizers "gradio>=6,<7"
%cd {REPO}
if REPO not in sys.path:
    sys.path.insert(0, REPO)
for m in ['model', 'tokenizer_utils', 'chat', 'web_chat']:
    sys.modules.pop(m, None)  # забываем старые версии кода
# 3) загружаем модель и открываем чат
from chat import ChatBot
from web_chat import launch
model_path = 'model/model.pt'
if модель.startswith('ролевая') and os.path.exists('model/model_rp.pt'):
    model_path = 'model/model_rp.pt'
if модель.startswith('большая'):
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE = '/content/drive/MyDrive/ai_training'
    # готовая модель, а если обучение ещё идёт — последнее сохранение
    for p in (f'{SAVE}/model_medium.pt', f'{SAVE}/ckpt/full.pt'):
        if os.path.exists(p):
            model_path = p
            break
    else:
        print('⚠️ Большой модели на Диске пока нет — запускаю маленькую.')
bot = ChatBot(model_path)
print('Модель загружена:', model_path, '| параметров: %.0f млн' % (bot.model.num_params() / 1e6), '| шаг обучения:', bot.ckpt['iter'])
print('👇 Открой ссылку «public URL» ниже')
launch(share=True, bot=bot)

---
### Дополнительно (не обязательно)
Работают после ячейки «Запустить чат».

In [ ]:
#@title Задать один вопрос без интерфейса
вопрос = "Как приготовить блины?" #@param {type:"string"}
температура = 0.7 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
if 'bot' not in globals() or not hasattr(bot, 'stream'):
    print('⚠️ Сначала нажми ▶ у ячейки «Запустить чат» (самая первая).')
else:
    bot.temperature = температура
    bot.reset()
    print('Ты: ', вопрос)
    print('Бот:', bot.answer(вопрос))